In [1]:
# Import libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create model
model = LogisticRegression(max_iter=200)

# Train model
model.fit(X_train, y_train)

# Test model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

# Save model
joblib.dump(model, "model.pkl")

print("Model saved successfully!")

Model Accuracy: 1.0
Model saved successfully!


In [2]:
%%writefile app.py

from flask import Flask, render_template, request
import joblib

app = Flask(__name__)

# Load trained model
model = joblib.load("model.pkl")

# Flower names
flower_names = [
    "Iris Setosa",
    "Iris Versicolor",
    "Iris Virginica"
]

@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():

    # Get values from form
    sepal_length = float(request.form["sepal_length"])
    sepal_width = float(request.form["sepal_width"])
    petal_length = float(request.form["petal_length"])
    petal_width = float(request.form["petal_width"])

    # Make prediction
    prediction = model.predict([[
        sepal_length,
        sepal_width,
        petal_length,
        petal_width
    ]])

    result = flower_names[prediction[0]]

    return render_template(
        "index.html",
        prediction=result
    )


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Writing app.py


In [3]:
!mkdir -p templates

In [4]:
%%writefile templates/index.html

<!DOCTYPE html>
<html>

<head>

    <title>Iris Flower Prediction</title>

    <style>

        body {
            font-family: Arial;
            background: #f2f2f2;
            text-align: center;
            padding: 40px;
        }

        .container {
            background: white;
            width: 400px;
            margin: auto;
            padding: 30px;
            border-radius: 15px;
            box-shadow: 0px 0px 10px #aaa;
        }

        input {
            width: 90%;
            padding: 10px;
            margin: 8px;
        }

        button {
            width: 95%;
            padding: 12px;
            background: #333;
            color: white;
            border: none;
            border-radius: 5px;
        }

        h1 {
            margin-bottom: 20px;
        }

        .result {
            margin-top: 20px;
            font-size: 20px;
            font-weight: bold;
        }

    </style>

</head>

<body>

<div class="container">

    <h1>Iris Flower Prediction</h1>

    <form action="/predict" method="POST">

        <input
            type="number"
            step="any"
            name="sepal_length"
            placeholder="Sepal Length"
            required
        >

        <input
            type="number"
            step="any"
            name="sepal_width"
            placeholder="Sepal Width"
            required
        >

        <input
            type="number"
            step="any"
            name="petal_length"
            placeholder="Petal Length"
            required
        >

        <input
            type="number"
            step="any"
            name="petal_width"
            placeholder="Petal Width"
            required
        >

        <button type="submit">
            Predict Flower
        </button>

    </form>

    {% if prediction %}

        <div class="result">
            Prediction: {{ prediction }}
        </div>

    {% endif %}

</div>

</body>

</html>

Writing templates/index.html


In [5]:
%%writefile requirements.txt

Flask
gunicorn
scikit-learn
joblib
numpy

Writing requirements.txt
